# Project: Road Traffic Accidents Analysis in Finland
## Framework: CRISP-DM Process Model

### 1. Business Understanding


The objective of this project is to investigate whether environmental and seasonal factors are associated with the severity of road accidents in Finland. We combine two different datasets; road accident data provided by Statistics Finland (Tilastokeskus) and weather data provided by Finnish Meteorological Institute (FMI).

We use **Logistic Regression** as the main machine learning method because the target variable has two possible outcomes. The model is used to analyze which environmental and weather factors are associated with accident severity and how well can the model predict whether an accident is severe or less severe.

## 2. Data Understanding


### 2.1 Setup


The project uses data from two sources. The road traffic accident data from **Statistics Finland** is provided as yearly CSV files covering the period from 2015 to 2024. The available files are loaded from the local datasets directory and combined into a single DataFrame.


The weather data is retrieved from the **Finnish Meteorological Institute (FMI) Open Data service** using the stored query. Daily observations on the weather are collected from the weather stations across the Finland. The analysis focuses on daily mean temperatures, precipitation and snow depth.

To make the data collection more efficient, the downloaded **FMI data** is stored locally under the **datasets/fmi_cache/**. If a file is already available, is it loaded from the cache instead of being downloaded again.

At this stage, **the two datasets** are loaded and examined separately. Their structures, variables, datatypes and missing values are inspected before proceeding to data preparation and integration.

In [101]:
# 1. Install and verify required spatial and ML libraries
# %pip install geopandas contextily requests fmiopendata mlxtend scikit-learn

# 2. Import core engines
import glob
import os
import time
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import contextily as ctx
import datetime as dt

from fmiopendata.wfs import download_stored_query

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, PoissonRegressor
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (mean_absolute_error, roc_auc_score, average_precision_score,
                             classification_report, ConfusionMatrixDisplay, RocCurveDisplay)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)

RANDOM_STATE = 69
DATA_DIR = "datasets"            # folder that holds tieliikenne_YYYY.csv
YEARS = list(range(2015, 2025))
TEST_YEARS = [2023, 2024]        # final hold-out: the two most recent years (time-based split)
print("Libraries imported.")

Libraries imported.


In [102]:
# 3. Data Loading - Correcting the separator to comma ',' as per actual data structure
#csv_path = 'datasets/tieliikenne_2024.csv'
#df_accidents = pd.read_csv(csv_path, sep=',')


import glob

# Load and combine all CSV files matching the pattern
accident_files = [os.path.join(DATA_DIR, f"tieliikenne_{year}.csv") for year in YEARS]
df_list = [pd.read_csv(f, sep=',') for f in accident_files if os.path.exists(f)]

df_accidents = pd.concat(df_list, ignore_index=True)

print(f"Loaded {len(df_list)} accident files.")
print("Total Shape (rows, columns):", df_accidents.shape)

Loaded 10 accident files.
Total Shape (rows, columns): (38372, 17)


In [103]:
print("Shape (rows, columns):", df_accidents.shape)
print("\nColumns:", list(df_accidents.columns))
print("\nData types:")
print(df_accidents.dtypes)

Shape (rows, columns): (38372, 17)

Columns: ['FID', 'id', 'geom', 'vvonn', 'kkonn', 'kello', 'vakav', 'onntyyppi', 'lkmhapa', 'lkmlaka', 'lkmjk', 'lkmpp', 'lkmmo', 'lkmmp', 'lkmmuukulk', 'x', 'y']

Data types:
FID               str
id              int64
geom              str
vvonn           int64
kkonn           int64
kello             str
vakav           int64
onntyyppi       int64
lkmhapa         int64
lkmlaka         int64
lkmjk           int64
lkmpp           int64
lkmmo           int64
lkmmp           int64
lkmmuukulk      int64
x             float64
y             float64
dtype: object


In total, there are 17 columns of data in the dataset. 

In [104]:
print("\nMissing values per column:")
print(df_accidents.isna().sum())


Missing values per column:
FID            0
id             0
geom           0
vvonn          0
kkonn          0
kello         16
vakav          0
onntyyppi      0
lkmhapa        0
lkmlaka        0
lkmjk          0
lkmpp          0
lkmmo          0
lkmmp          0
lkmmuukulk     0
x              0
y              0
dtype: int64


There is only one missing data for **kello**.

### 2.4 Weather data (Finnish Meteorological Institute)
**Source:** FMI open data WFS, stored query `fmi::observations::weather::daily::multipointcoverage` (daily observations from all weather stations in the bounding box of Finland).

**Variables requested**

| FMI code | Meaning |
|---|---|
| `tday` | daily mean temperature (°C) |
| `rrday` | daily precipitation (mm) |
| `snow` | snow depth (cm) |

**How the download works and why**
- The query is sent **one calendar month at a time**. A request for a whole year over all of Finland is too large for the service and fails or times out.
- Each downloaded month is saved to `datasets/fmi_cache/`. If a download is interrupted or a month fails, just **run the cell again**: finished months are read from disk, only missing ones are fetched.
- Parameter names are matched by keyword (`temperature`, `precip`, `snow`) and the function stops with a clear message if the temperature cannot be found, so a naming change in the FMI service cannot silently produce an empty dataset.
- If a full-month request fails, the function automatically retries it as two half-month requests.

In [105]:
FMI_QUERY = "fmi::observations::weather::daily::multipointcoverage"
FMI_BBOX = "19,59,32,71"            # lon_min, lat_min, lon_max, lat_max = all of Finland
FMI_PARAMS = "tday,rrday,snow"      # mean temperature, precipitation, snow depth
WEATHER_DIR = os.path.join(DATA_DIR, "fmi_cache")
os.makedirs(WEATHER_DIR, exist_ok=True)


def _find_value(params, *keywords):
    # Return the value of the first FMI parameter whose name contains one of the keywords.
    for name, item in params.items():
        if any(k in name.lower() for k in keywords):
            value = item.get("value")
            return np.nan if value is None else float(value)
    return np.nan


def _download_range(start, end, retries=3, pause=5):
    # Download one time range from FMI with retries. Returns obs.data or None.
    for attempt in range(1, retries + 1):
        try:
            result = download_stored_query(
                FMI_QUERY,
                args=[f"bbox={FMI_BBOX}",
                      f"starttime={start:%Y-%m-%dT%H:%M:%SZ}",
                      f"endtime={end:%Y-%m-%dT%H:%M:%SZ}",
                      f"parameters={FMI_PARAMS}"],
            )
            ## TODO: aikaisemmin if result.data: palautetaan kaikki data
            if result:
                return result.data, result.location_metadata
            reason = "empty response"
        except Exception as err:
            reason = f"{type(err).__name__}: {str(err)[:80]}"
        print(f"[{reason}; attempt {attempt}]", end=" ")
        time.sleep(pause * attempt)
    return None


def fetch_fmi_month(year, month):
    # Daily weather for one calendar month -> DataFrame(date, station, tday, rrday, snow, location).
    start = pd.Timestamp(year, month, 1)
    end = start + pd.offsets.MonthBegin(1)

    data_parts = []
    metadata_parts = []
    data, location_metadata = _download_range(start, end) # TODO: this is new
    if data is not None:
        data_parts.append(data)
        metadata_parts.append(location_metadata) # TODO: this is new

    else:                                   # fall back to two half-month requests
        middle = start + pd.Timedelta(days=15)
        for a, b in [(start, middle), (middle, end)]:
            part, metadata_part = _download_range(a, b, retries=2)
            if part is None:
                return None
            data_parts.append(part)
            metadata_parts.append(metadata_part) # TODO: this is new

    # TODO: this is new
    # flatten list of dicts
    location_dict = {}
    for location_datas in metadata_parts:
        for key, value in location_metadata.items():
            location_dict[key] = value


    rows, example_keys = [], None
    for data in data_parts:
        for timestamp, stations in data.items():
            for station, params in stations.items():
                example_keys = example_keys or list(params.keys())
                rows.append({
                    "date": pd.Timestamp(timestamp).normalize(),
                    "station": station,
                    "tday": _find_value(params, "temperature"),
                    "rrday": _find_value(params, "precip", "rain"),
                    "snow": _find_value(params, "snow"),
                    "location" : location_dict[station] # TODO: this is new
                })
    df = pd.DataFrame(rows)
    if df["tday"].isna().all():
        raise RuntimeError("Temperature not found in the FMI response. "
                           f"Parameter names returned by FMI: {example_keys}")

    return df.drop_duplicates(["date", "station"])


def load_weather_month(year, month):
    # Read a month from the local cache or download it (and cache it).
    path = os.path.join(WEATHER_DIR, f"fmi_daily_{year}-{month:02d}.csv")
    if os.path.exists(path):
        return pd.read_csv(path, parse_dates=["date"])
    print(f"FMI {year}-{month:02d}: downloading ...", end=" ")
    part = fetch_fmi_month(year, month)
    if part is None:
        print("FAILED")
        return None
    part.to_csv(path, index=False)
    print(f"{len(part)} rows, {part['station'].nunique()} stations")
    return part

In [106]:
weather_parts, failed_months = [], []
for year in YEARS:
    for month in range(1, 13):
        part = load_weather_month(year, month)
        if part is None:
            failed_months.append(f"{year}-{month:02d}")
        else:
            weather_parts.append(part)

if failed_months:
    raise RuntimeError(f"{len(failed_months)} months could not be downloaded: {failed_months}. "
                       "Run this cell again - finished months are cached and will not be re-downloaded.")

df_weather_daily = pd.concat(weather_parts, ignore_index=True).drop_duplicates(["date", "station"])
print("Daily station observations:", df_weather_daily.shape)
display(df_weather_daily.head())

Daily station observations: (994322, 6)


,date,station,tday,rrday,snow,location
0,2015-01-01,Jomala Maarianhamina lentoasema,4.3,NaN,NaN,"{'fmisid': 100907, 'latitude': 60.12467, 'long..."
1,2015-01-01,Parainen Utö,4.6,5.0,-1.0,"{'fmisid': 100908, 'latitude': 59.77909, 'long..."
2,2015-01-01,Lemland Nyhamn,4.7,NaN,NaN,"{'fmisid': 100909, 'latitude': 59.95911, 'long..."
3,2015-01-01,Jomala Jomalaby,3.4,8.5,5.0,"{'fmisid': 100917, 'latitude': 60.17824, 'long..."
4,2015-01-01,Hammarland Märket,3.7,NaN,NaN,"{'fmisid': 100919, 'latitude': 60.30098, 'long..."


In [107]:
# KORJATTU LAATUTARKASTUS: Rajataan data tiukasti vuosiin 2015-2024
w = df_weather_daily.copy()

# MASTER FIX: Poistetaan vuoden 2016 vuotanut ylimääräinen päivä, jotta kuukausien määrä on tasan 120
w['year_check'] = w['date'].dt.year
w = w[w['year_check'].isin(YEARS)]

# Lasketaan kattavuus uudestaan siivotusta datasta
cover = (w.assign(vuosi=w["date"].dt.year, kuukausi=w["date"].dt.month)
          .groupby(["vuosi", "kuukausi"])
          .agg(days=("date", "nunique"), stations=("station", "nunique")))

print("--- FMI Säädatan Laatutarkastus ---")
print("Months covered:", len(cover), "(odotettu: 120)")
print("Days with data per month  : min", cover["days"].min(), "| max", cover["days"].max())
print("Stations per month        : min", cover["stations"].min(), "| max", cover["stations"].max())
print("\nShare of missing values:")
print(w[["tday", "rrday", "snow"]].isna().mean().round(3))
print("\nValue ranges:")
display(w[["tday", "rrday", "snow"]].describe().T)

# Nyt nämä assertit menevät heittämällä läpi ilman virheitä!
assert len(cover) == 120, "Some months are missing from the weather data"
assert cover["days"].min() >= 28, "A month has fewer than 28 days of observations"
print("\n Kaikki tarkastukset menivät läpi! Data on puhdas ja kuukausia on tasan 120!")


--- FMI Säädatan Laatutarkastus ---
Months covered: 120 (odotettu: 120)
Days with data per month  : min 28 | max 31
Stations per month        : min 266 | max 291

Share of missing values:
tday     0.284
rrday    0.292
snow     0.320
dtype: float64

Value ranges:


,count,mean,std,min,25%,50%,75%,max
tday,711439.0,4.320257,9.598106,-42.4,-1.6,3.9,12.2,27.6
rrday,703686.0,1.302800,4.107019,-1.0,-1.0,0.1,1.8,115.3
snow,675796.0,13.845866,24.088720,-1.0,-1.0,-1.0,23.0,143.0



 Kaikki tarkastukset menivät läpi! Data on puhdas ja kuukausia on tasan 120!


**Notes on the weather values**
- FMI codes a trace amount of precipitation (and "no snow") as **−1**. These values are set to 0 in the preparation phase.
- Snow depth is measured only at some stations and mostly in winter. In summer it is empty; we treat a month without any snow observation as 0 cm.
- The number of reporting stations changes over time, which is a source of small inconsistencies in the national average.

### 2.X Weather measurement locations




In [108]:
df_weather_daily

,date,station,tday,rrday,snow,location
0,2015-01-01,Jomala Maarianhamina lentoasema,4.3,NaN,NaN,"{'fmisid': 100907, 'latitude': 60.12467, 'long..."
1,2015-01-01,Parainen Utö,4.6,5.0,-1.0,"{'fmisid': 100908, 'latitude': 59.77909, 'long..."
2,2015-01-01,Lemland Nyhamn,4.7,NaN,NaN,"{'fmisid': 100909, 'latitude': 59.95911, 'long..."
3,2015-01-01,Jomala Jomalaby,3.4,8.5,5.0,"{'fmisid': 100917, 'latitude': 60.17824, 'long..."
4,2015-01-01,Hammarland Märket,3.7,NaN,NaN,"{'fmisid': 100919, 'latitude': 60.30098, 'long..."
...,...,...,...,...,...,...
1026710,2025-01-01,Kittilä Lompolonvuoma,-16.6,-1.0,NaN,NaN
1026711,2025-01-01,Kuusamo Ruka Talvijärvi,-8.3,NaN,NaN,NaN
1026712,2025-01-01,Espoo Nuuksio,-3.6,4.9,5.0,NaN
1026713,2025-01-01,Mikkeli Lentoasema AWOS,-6.1,3.3,18.0,NaN


### 3. Data Preparation
Since raw accident records represent single events, direct continuous linear modeling on row levels is mathematically invalid. We perform an **Aggregation step** to compute total accidents grouped by month, establishing a true continuous numeric target variable suitable for Linear Regression modeling.

### 3.1 Accident data: readable names, hour of day, accident type, target

In [109]:
# Readable column names for presentation
readable_col_names =  {
"vvonn" : "vuosi",
"kkonn" : "kuukausi",
"kello" : "kello",
"vakav" : "vakavuus",
"onntyyppi" : "onnettomuustyyppi",
"lkmhapa" : "henkilö- ja pakettiautot",
"lkmlaka" : "linja- ja kuorma-autot",
"lkmjk" : "jalankulkijat",
"lkmpp" : "polkupyörät ja kevyet sähköajoneuvot",
"lkmmo" : "mopot",
"lkmmp" : "moottoripyörät",
"lkmmuukulk" : "muut kulkuneuvot",
"x" : "x",
"y" : "y"
}
#
df_accidents.rename( columns=readable_col_names, inplace=True)

### Converting class variables to 1-hot encoded columns

In [110]:
accident_types_raw = {
0 : "samat ajosuunnat (ajo suoraan)",
1 : "samat ajosuunnat (ajo kääntyen)",
2 : "vastakkaiset ajosuunnat (ajo suoraan)",
3 : "vastakkaiset ajosuunnat (ajo kääntyen)",
4 : "risteävät ajosuunnat (ajo suoraan)",
5 : "risteävät ajosuunnat (ajo kääntyen)",
6 : "jalankulkijaonnettomuus (suojatiellä)",
7 : "jalankulkijaonnettomuus (muualla)",
8 : "tieltä suistuminen",
9 : "muu onnettomuus"
}

accident_types_columns = {
0 : "type0",
1 : "type1",
2 : "type2",
3 : "type3",
4 : "type4",
5 : "type5",
6 : "type6",
7 : "type7",
8 : "type8",
9 : "type9"
}
# Create columns 'typeX' and assign boolean 1 or 0
for type_number, column_name in accident_types_raw.items():
    df_accidents[column_name] = [1 if accident_type == type_number else 0 for accident_type in df_accidents['onnettomuustyyppi']]

# Drop onnettomuustyyppi column
df_accidents.drop('onnettomuustyyppi', axis=1, inplace=True)

df_accidents.head(1)


,FID,id,geom,vuosi,kuukausi,kello,vakavuus,henkilö- ja pakettiautot,linja- ja kuorma-autot,jalankulkijat,polkupyörät ja kevyet sähköajoneuvot,mopot,moottoripyörät,muut kulkuneuvot,x,y,samat ajosuunnat (ajo suoraan),samat ajosuunnat (ajo kääntyen),vastakkaiset ajosuunnat (ajo suoraan),vastakkaiset ajosuunnat (ajo kääntyen),risteävät ajosuunnat (ajo suoraan),risteävät ajosuunnat (ajo kääntyen),jalankulkijaonnettomuus (suojatiellä),jalankulkijaonnettomuus (muualla),tieltä suistuminen,muu onnettomuus
0,tieliikenne_2015.1,1,POINT (397544.8774 6675865.1006),2015,1,18.00-18.59,2,1,0,1,0,0,0,0,397544.8774,6.675865e+06,0,0,0,0,0,0,1,0,0,0


In [111]:
# Convert accident lethality to 0 or 1
#df_accidents['vakavuus'] = [1 if vakavuus == 1 else 0 for vakavuus in df_accidents['vakavuus']]

df_accidents['vakavuus'] = df_accidents['vakavuus'].isin([1, 3]).astype(int)
df_accidents

,FID,id,geom,vuosi,kuukausi,kello,vakavuus,henkilö- ja pakettiautot,linja- ja kuorma-autot,jalankulkijat,polkupyörät ja kevyet sähköajoneuvot,mopot,moottoripyörät,muut kulkuneuvot,x,y,samat ajosuunnat (ajo suoraan),samat ajosuunnat (ajo kääntyen),vastakkaiset ajosuunnat (ajo suoraan),vastakkaiset ajosuunnat (ajo kääntyen),risteävät ajosuunnat (ajo suoraan),risteävät ajosuunnat (ajo kääntyen),jalankulkijaonnettomuus (suojatiellä),jalankulkijaonnettomuus (muualla),tieltä suistuminen,muu onnettomuus
0,tieliikenne_2015.1,1,POINT (397544.8774 6675865.1006),2015,1,18.00-18.59,0,1,0,1,0,0,0,0,397544.8774,6.675865e+06,0,0,0,0,0,0,1,0,0,0
1,tieliikenne_2015.2,2,POINT (382175.8528 6678657.1592),2015,1,06.00-06.59,0,1,0,1,0,0,0,0,382175.8528,6.678657e+06,0,0,0,0,0,0,1,0,0,0
2,tieliikenne_2015.3,3,POINT (393635.8546 6677840.6594),2015,1,16.00-16.59,0,3,0,0,0,0,0,0,393635.8546,6.677841e+06,1,0,0,0,0,0,0,0,0,0
3,tieliikenne_2015.4,4,POINT (383847.8421 6671277.6964),2015,12,12.00-12.59,0,1,0,0,1,0,0,0,383847.8421,6.671278e+06,0,0,0,0,1,0,0,0,0,0
4,tieliikenne_2015.5,5,POINT (386309.4585 6673319.5558),2015,1,07.00-07.59,0,1,0,1,0,0,0,0,386309.4585,6.673320e+06,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38367,tieliikenne_2024.2805,2805,POINT (126484.77840000018 6682647.3047),2024,11,12.00-12.59,0,0,1,0,0,0,0,0,126484.7784,6.682647e+06,0,0,0,0,0,0,0,0,1,0
38368,tieliikenne_2024.2806,2806,POINT (107800.28100000042 6683502.192500001),2024,12,17.00-17.59,0,1,0,0,0,0,0,0,107800.2810,6.683502e+06,0,0,0,0,0,0,0,0,1,0
38369,tieliikenne_2024.2807,2807,POINT (116624.37719999999 6701750.947699999),2024,12,15.00-15.59,0,3,0,0,0,0,0,0,116624.3772,6.701751e+06,0,1,0,0,0,0,0,0,0,0
38370,tieliikenne_2024.2808,2808,POINT (110664.54220000003 6684589.5315000005),2024,12,08.00-08.59,0,2,0,0,0,0,0,0,110664.5422,6.684590e+06,1,0,0,0,0,0,0,0,0,0


### Converting cyclical time variable to linear difference from midday

The hours are are in String datatype and in format XX.00-XX.59.

We take the first two characters of each string in column 'kello' and convert that to float. We convert it into float as there is a one missing value **(NaN)** that cannot be an Integer.

After dropping the missing row, conversion to whole numbers is done with **.astype(int)**

In [112]:
df_accidents['tunti'] = df_accidents['kello'].str[:2].astype(float)

# drop the row(s) with missing time, then make it an integer
df_accidents = df_accidents.dropna(subset=['tunti'])
df_accidents['tunti'] = df_accidents['tunti'].astype(int)

# Subtracts each accident's hour from 12 and takes the absolute value
df_accidents['tunti'] = (12 - df_accidents['tunti']).abs()

# Drop the column 'kello'.
df_accidents = df_accidents.drop(columns=['kello'])

In [113]:
# Spatial Transformation - Convert to GeoDataFrame using Finnish coordinate system (EPSG:3067)
accidents_gdf = gpd.GeoDataFrame(
    df_accidents,
    geometry=gpd.points_from_xy(df_accidents.x, df_accidents.y),
    crs="EPSG:3067"
)

print(" Data loaded and transformed successfully into Spatial GeoDataFrame!")
display(accidents_gdf.head())

 Data loaded and transformed successfully into Spatial GeoDataFrame!


,FID,id,geom,vuosi,kuukausi,vakavuus,henkilö- ja pakettiautot,linja- ja kuorma-autot,jalankulkijat,polkupyörät ja kevyet sähköajoneuvot,mopot,moottoripyörät,muut kulkuneuvot,x,y,samat ajosuunnat (ajo suoraan),samat ajosuunnat (ajo kääntyen),vastakkaiset ajosuunnat (ajo suoraan),vastakkaiset ajosuunnat (ajo kääntyen),risteävät ajosuunnat (ajo suoraan),risteävät ajosuunnat (ajo kääntyen),jalankulkijaonnettomuus (suojatiellä),jalankulkijaonnettomuus (muualla),tieltä suistuminen,muu onnettomuus,tunti,geometry
0,tieliikenne_2015.1,1,POINT (397544.8774 6675865.1006),2015,1,0,1,0,1,0,0,0,0,397544.8774,6.675865e+06,0,0,0,0,0,0,1,0,0,0,6,POINT (397544.877 6675865.101)
1,tieliikenne_2015.2,2,POINT (382175.8528 6678657.1592),2015,1,0,1,0,1,0,0,0,0,382175.8528,6.678657e+06,0,0,0,0,0,0,1,0,0,0,6,POINT (382175.853 6678657.159)
2,tieliikenne_2015.3,3,POINT (393635.8546 6677840.6594),2015,1,0,3,0,0,0,0,0,0,393635.8546,6.677841e+06,1,0,0,0,0,0,0,0,0,0,4,POINT (393635.855 6677840.659)
3,tieliikenne_2015.4,4,POINT (383847.8421 6671277.6964),2015,12,0,1,0,0,1,0,0,0,383847.8421,6.671278e+06,0,0,0,0,1,0,0,0,0,0,0,POINT (383847.842 6671277.696)
4,tieliikenne_2015.5,5,POINT (386309.4585 6673319.5558),2015,1,0,1,0,1,0,0,0,0,386309.4585,6.673320e+06,0,0,0,0,0,0,1,0,0,0,5,POINT (386309.458 6673319.556)


### More

In [114]:
# Drop id
df_accidents.drop('id', axis=1, inplace=True)

# Drop unnecessary string fields
df_accidents.drop('FID', axis=1, inplace=True)
df_accidents.drop('geom', axis=1, inplace=True)


# TODO: Delete this removal later, as of now 2024 holds no value
df_accidents.drop('vuosi', axis=1, inplace=True)
df_accidents.drop('kuukausi', axis=1, inplace=True)
df_accidents.drop('x', axis=1, inplace=True)
df_accidents.drop('y', axis=1, inplace=True)


### 3.2 Weather: from daily station values to one value per month
1. Clean the codes (−1 = trace → 0).
2. Average over all stations to get one national value **per day**.
3. Aggregate the days into **monthly features**: mean temperature, total precipitation, mean snow depth and the share of days with a mean temperature below 0 °C.

In [115]:
# 1. Create a copy of the raw daily weather table
d_stations = df_weather_daily.copy()

# Clean up trace amounts and missing codes as we did before
d_stations["rrday"] = d_stations["rrday"].clip(lower=0)
d_stations["snow"] = d_stations["snow"].clip(lower=0)

# Extract year and month fields from the datetime index
d_stations["vuosi"] = d_stations["date"].dt.year
d_stations["kuukausi"] = d_stations["date"].dt.month

# 2. Aggregation Phase: Group by Year, Month, Station Name, and Location coordinates!
# This collapses the 10-minute/daily records into a clean Monthly Station Dataset.
df_weather_monthly_stations = d_stations.groupby(
    ["vuosi", "kuukausi", "station", "location"], 
    as_index=False
).agg(
    keski_lampotila=("tday", "mean"),       # Monthly mean temp for THIS specific station
    sade_summa=("rrday", "sum"),            # Monthly total rain for THIS specific station
    lumensyvyys=("snow", "mean"),          # Monthly mean snow depth for THIS specific station
    pakkaspaivat_osuus=("tday", lambda s: (s.dropna() < 0).mean()) # Frost days share for THIS station
)

# Handle missing snow values for the stations safely
df_weather_monthly_stations["lumensyvyys"] = df_weather_monthly_stations["lumensyvyys"].fillna(0)

print("Monthly weather dataset with unique locations constructed successfully!")
print(f"Dataset Shape (Monthly Station Records): {df_weather_monthly_stations.shape}")

# Display the first 14 rows to let Abu Shahab see the station names and coordinates
display(df_weather_monthly_stations.head(14))


Monthly weather dataset with unique locations constructed successfully!
Dataset Shape (Monthly Station Records): (17420, 8)


,vuosi,kuukausi,station,location,keski_lampotila,sade_summa,lumensyvyys,pakkaspaivat_osuus
0,2015,1,Alajärvi Möksy,"{'fmisid': 101533, 'latitude': 63.08898, 'long...",-6.529032,107.4,34.482759,0.903226
1,2015,1,Alavus Sulkavankylä,"{'fmisid': 101305, 'latitude': 62.44881, 'long...",NaN,62.9,16.967742,NaN
2,2015,1,Asikkala Pulkkilanharju,"{'fmisid': 101185, 'latitude': 61.26521, 'long...",-3.761290,0.0,0.000000,0.741935
3,2015,1,Asikkala Urajärvi,"{'fmisid': 101168, 'latitude': 61.13694, 'long...",NaN,62.9,27.258065,NaN
4,2015,1,Enonkoski Simanala,"{'fmisid': 101446, 'latitude': 62.05563, 'long...",NaN,68.2,32.225806,NaN
5,2015,1,Enontekiö Hetta,"{'fmisid': 101974, 'latitude': 68.38606, 'long...",NaN,39.6,58.258065,NaN
6,2015,1,Enontekiö Kaaresuvanto,"{'fmisid': 101968, 'latitude': 68.44298, 'long...",NaN,38.0,51.677419,NaN
7,2015,1,Enontekiö Kilpisjärvi Saana,"{'fmisid': 102017, 'latitude': 69.04277, 'long...",-12.483871,0.0,0.000000,1.000000
8,2015,1,Enontekiö Kilpisjärvi kyläkeskus,"{'fmisid': 102016, 'latitude': 69.04948, 'long...",-14.693548,29.2,53.129032,0.967742
9,2015,1,Enontekiö Näkkälä,"{'fmisid': 102019, 'latitude': 68.60301, 'long...",-14.635484,15.2,55.225806,1.000000


Convert daily FMI observations into monthly averages
WHILE KEEPING the unique Station Name and its Geographic Location!


In [ ]:
WX_FEATURES = ["keski_lampotila", "sade_summa", "lumensyvyys", "pakkaspaivat_osuus"]

fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True)
x = pd.to_datetime(dict(year=df_weather_monthly["vuosi"], month=df_weather_monthly["kuukausi"], day=1))
titles = ["Mean temperature (°C)", "Total precipitation (mm, station average)",
          "Mean snow depth (cm)", "Share of days below 0 °C"]
for ax, col, title in zip(axes.ravel(), WX_FEATURES, titles):
    ax.plot(x, df_weather_monthly[col], color="teal")
    ax.set_title(title)
plt.tight_layout(); plt.show()

### 3.X Separating training and testing data

The data is divided into training (75%) and testing (25%). To guarantee same results across every run, a fixed **random_state** is applied.

In [ ]:
from sklearn.model_selection import train_test_split

y = df_accidents['vakavuus']
X = df_accidents.drop(columns= 'vakavuus')

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=69
)

In [ ]:
# TODO: You can remove this later
X_train

In [ ]:
# TODO: You can remove this later
X_test

In [ ]:
y_train

In [ ]:
y_test

## 3.X Feature scaling

To prepare the features for moding, we standardize them using **Scikit-Learn's StandardScaler**. This transforms the features to have a mean of 0 and a standard deviation of 1.

The scaling is done due to Logistic Regression is sensitive to the scale of input features. Features with larger numeric ranges could otherwise dominate the model's coefficients.

Scaler is fitted strictly on the training set **X_train** to learn its mean and variance. These values are then applied to the test set **X_test**, ensuring no information from the testing data leaks into the training process.

In [ ]:
## Scale features

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit the scaler on X_train data only to learn its mean and variance
X_train_scaled = scaler.fit_transform(X_train)
# Scale X_test using the exact same mean and variance learned from X_train
X_test_scaled = scaler.transform(X_test)

# Wrap results back into a DataFrame
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_train.columns)

# assert the list sizes, small check.
assert len(X_train_scaled) == len(y_train) and len(X_test_scaled) == len(y_test)

print("Data Preparation Completed:")
print(f"Training set size: {X_train.shape[0]} samples.")
print(f"Testing set size: {X_test.shape[0]} samples.")

print("\n1. Without standardize features train data :")
display(X_train.head(5))

print("\n2. With standardize features train data (for the Logistic regression) :")
display(X_train_scaled.head(5))



# TODO: ADD REGRESSION MODELING AND VALIDATION BELLOW

### 4. Modeling

With the data prepared, we initialize and train a **LogisticRegression** model from Scikit-Learn. The model is train using the **X_train_scaled** and **y_train**.


In [ ]:
from sklearn.linear_model import LogisticRegression

# Model initialization and training
model = LogisticRegression(class_weight='balanced', random_state=69)
model.fit(X_train_scaled, y_train)

In [ ]:
# TODO: Move to 5. Evaluation later

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# Train accuracy
print("Train accuracy:", model.score(X_train_scaled, y_train))
# Test accuracy
print("Test accuracy: ", model.score(X_test_scaled, y_test))

In [ ]:
pred = model.predict(X_test_scaled)
print(classification_report(y_test, pred))

ConfusionMatrixDisplay.from_predictions(
    y_test, pred,
    display_labels=['Less severe', 'Severe'],
    cmap='Blues'
)
plt.title('Logistic regression: confusion matrix')
plt.show()

In [ ]:
print(y_train.value_counts())
print(y_test.value_counts())
print(df_accidents['vakavuus'].value_counts())

### 4. X Alternative Cross Validation
Cross validation instead of split validation

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score

# cross-validation

y_pred = cross_val_predict(estimator=model, X=X_train_scaled, y=y_train, cv=10)

cm = confusion_matrix(y_train, y_pred)
accuracy = accuracy_score(y_train, y_pred)

print("Accuracy: %0.2f" % accuracy)
print("Confusion Matrix:\n", cm)

# visualize confusion matrix
plt.matshow(cm)
plt.title('Confusion matrix')
plt.colorbar()
plt.ylabel('True label')
plt.xlabel('Predicted label')
# include counts
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha='center', va='center', color='red', backgroundcolor='white')
plt.show()

### 4.x FMI Weather Interface Testing
We test the FMI Open Data live streaming engine to collect auxiliary meteorological variables corresponding to the accident location spaces.

In [ ]:
# TODO: this is for testing weather data only

import pandas as pd
import datetime as dt
from fmiopendata.wfs import download_stored_query

try:
    print("Connecting to FMI Government server... Fetching raw spatial coordinates...")
    
    # 1. Fetch data from FMI using a defined bounding box (Standard multipoint query)
    obs = download_stored_query(
        "fmi::observations::weather::multipointcoverage", 
        args=["bbox=24.5,60.1,25.1,60.3"]
    )
    
    # 2. Manual dictionary parsing loop to completely bypass 'to_dataframe' error
    weather_records = []
    
    # Loop through each unique timestamp key in the raw data
    for timestamp in obs.data.keys():
        # Loop through each weather station name key at this timestamp
        for station_name in obs.data[timestamp].keys():
            station_data = obs.data[timestamp][station_name]
            
            # Extract main target parameters provided by FMI
            temp = station_data.get('Air temperature', {}).get('value', None)
            wind = station_data.get('Wind speed', {}).get('value', None)
            rh = station_data.get('Relative humidity', {}).get('value', None)
            
            # Append rows manually to our collection list
            weather_records.append({
                "time": timestamp,
                "location": station_name,
                "air_temperature": temp,
                "wind_speed": wind,
                "relative_humidity": rh
            })
            
    # 3. Construct the clean Pandas DataFrame manually!
    weather_df = pd.DataFrame(weather_records)
    
    print("Real FMI Weather database constructed successfully without errors!")
    print(f"Weather Data Shape: {weather_df.shape}")
    
    # Display the first 10 rows for data understanding validation
    display(weather_df.head(10))

except Exception as e:
    print(f"Final Engine Error Notice: {e}. Check server status.")


### 5. Evaluation

### 5.X Feature Importance

In [ ]:
from sklearn.inspection import permutation_importance

# Calculate permutation importance using the standardized model and scaled test data
result = permutation_importance(
    model,
    X_train_scaled,
    y_train,
    n_repeats=10, # TODO: what is good number of repeats?
    scoring="accuracy",
    random_state=69,
    n_jobs=-1
)

# Sort features by their mean importance score
sorted_idx = result.importances_mean.argsort()

# Plot the results
fig, ax = plt.subplots()
ax.boxplot(
    result.importances[sorted_idx].T,
    orientation="horizontal",
    tick_labels=X_train_scaled.columns[sorted_idx] #aikaisemmin df_accidents
)
ax.set_title("Permutation Feature Importance")
fig.tight_layout()
plt.show()


### 6. Deployment